## Quickbooks Desktop Data Retrivel Using QODBC

### This script is designed to do the following:
1. Read configuration settings from a YAML file.
2. Connect to a database using a DSN (Database Source Name).
3. Fetch specified tables from the database as Pandas DataFrames.
4. Save each DataFrame as a Parquet file.
5. Upload the Parquet files to an AWS S3 bucket.


In [ ]:
import pyodbc
import pandas as pd
import boto3
import pyarrow.parquet as pq
import os
import yaml

#Step 1. Read configuration settings from a YAML file.
def read_config(yaml_file):
    with open(yaml_file, 'r') as file:
        config = yaml.safe_load(file)
    return config

#Step2. Connect to a database using a DSN and fetch a table as a DataFrame
def dsn_connection(table_name, connection):
    try:
        sql_query = f"SELECT * FROM {table_name}"
        df = pd.read_sql(sql_query, connection)
        return df
    except Exception as e:
        print(f"Error reading {table_name} table: {e}")
        return None

# Function to upload a Parquet file to an AWS S3 bucket
def upload_parquet_to_s3(aws_access_key_id, aws_secret_access_key, bucket_name, file_name, local_parquet_file_path):
    try:
        s3 = boto3.client('s3', aws_access_key_id=aws_access_key_id, aws_secret_access_key=aws_secret_access_key)
        with open(local_parquet_file_path, 'rb') as local_file:
            s3.upload_fileobj(local_file, bucket_name, file_name)
        print(f"{file_name} file uploaded to S3 bucket")
    except Exception as e:
        print(f"Error uploading {file_name} to S3 bucket: {e}")

# Load configuration from YAML file
config = read_config('config.yaml')

# Database connection parameters
dsn_name = config['database']['dsn_name']

# AWS S3 credentials and bucket name
aws_access_key_id = config['aws']['access_key_id']
aws_secret_access_key = config['aws']['secret_access_key']
bucket_name = config['aws']['bucket_name']

# List of tables to process
table_names = config['tables']

try:
    # Establish database connection
    connection = pyodbc.connect(f'DSN={dsn_name}', autocommit=True)
    cursor = connection.cursor()

    for table_name in table_names:
        # Fetch data from the database
        df = dsn_connection(table_name, connection)

#Step 4. Save each DataFrame as a Parquet file.
        if df is not None:
            parquet_file_path = f'{table_name}.parquet'
            df.to_parquet(parquet_file_path)
            print(f"'{table_name}' table data saved as Parquet file: {parquet_file_path}")

#Step5. Upload the Parquet files to an AWS S3 bucket.
            upload_parquet_to_s3(aws_access_key_id, aws_secret_access_key, bucket_name, parquet_file_path, parquet_file_path)
        else:
            print(f"Skipping '{table_name}' due to previous error.")

finally:
    # Close cursor and connection
    cursor.close()
    connection.close()
